In [2]:
import os
import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
    roc_curve, precision_recall_curve
)

# ============================================================
# CONFIGURATION
# ============================================================
class Config:
    DATA_DIR = "pcam_dataset"

    TEST_X = os.path.join(DATA_DIR, "camelyonpatch_level_2_split_test_x.h5")
    TEST_Y = os.path.join(DATA_DIR, "camelyonpatch_level_2_split_test_y.h5")

    SAVE_DIR = "opdense121att"
    os.makedirs(SAVE_DIR, exist_ok=True)

    BATCH_SIZE = 128
    NUM_WORKERS = 2
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    BEST_MODEL_PATH = "experiments/best_model_fold_4-Copy1.pth"


config = Config()

# ============================================================
# DATASET
# ============================================================
class PCamDataset(Dataset):
    def __init__(self, x_path, y_path, transform=None):
        self.x_path = x_path
        self.y_path = y_path
        self.transform = transform

        self.x_file = None
        self.x_data = None

        with h5py.File(y_path, "r") as f:
            self.labels = f["y"][:].reshape(-1)

        with h5py.File(x_path, "r") as f:
            self.length = f["x"].shape[0]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if self.x_file is None:
            self.x_file = h5py.File(self.x_path, "r")
            self.x_data = self.x_file["x"]

        img = self.x_data[idx].astype("float32") / 255.0
        label = float(self.labels[idx])

        if self.transform:
            img = (img * 255).astype(np.uint8)
            from PIL import Image
            img = Image.fromarray(img)
            img = self.transform(img)
        else:
            img = torch.tensor(img).permute(2, 0, 1)

        return img, label


def get_transforms(is_training=False):
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])


# ============================================================
# CBAM ATTENTION MODULES (MATCH EXACT TRAINING)
# ============================================================
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.fc(self.avg_pool(x).view(b,c))
        max_out = self.fc(self.max_pool(x).view(b,c))
        out = self.sigmoid(avg_out + max_out).view(b,c,1,1)
        return x * out


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2,1,kernel_size,padding=kernel_size//2,bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max_, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg, max_], dim=1)
        att = self.sigmoid(self.conv(out))
        return x * att


class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention()

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


# ============================================================
# FINAL MODEL (EXACT MATCH TO TRAINING)
# ============================================================
class DenseNetAttention(nn.Module):
    """DenseNet121 with CBAM (same as training checkpoint)"""
    def __init__(self, num_classes=1, dropout=0.5, pretrained=False):
        super(DenseNetAttention, self).__init__()

        self.backbone = models.densenet121(weights=None)
        num_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()

        # THE NAME MUST MATCH TRAINING: "attention"
        self.attention = CBAM(num_features, reduction=16)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(num_features,512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(512,256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout*0.5),

            nn.Linear(256,1)
        )

    def forward(self,x):
        f = self.backbone.features(x)
        f = self.attention(f)
        f = self.pool(f)
        f = f.view(f.size(0), -1)
        return self.classifier(f)


# ============================================================
# LOAD MODEL CHECKPOINT
# ============================================================
print("\nLoading model:", config.BEST_MODEL_PATH)

model = DenseNetAttention()
checkpoint = torch.load(config.BEST_MODEL_PATH, map_location=config.DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(config.DEVICE)
model.eval()

print("✓ Model loaded successfully!\n")


# ============================================================
# LOAD TEST LOADER
# ============================================================
test_dataset = PCamDataset(
    config.TEST_X,
    config.TEST_Y,
    transform=get_transforms(False)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

print("✓ Test Loader Ready. Samples =", len(test_dataset))


# ============================================================
# INFERENCE
# ============================================================
all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(config.DEVICE)

        outputs = model(images).squeeze()
        probs = torch.sigmoid(outputs).cpu().numpy()

        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

print("\n✓ Inference Complete!")


# ============================================================
# THRESHOLD OPTIMIZATION (MAX AUC + MAX ACC)
# ============================================================
thresholds = np.linspace(0.05, 0.95, 181)
best_thresh = 0.5
best_score = -1

base_auc = roc_auc_score(all_labels, all_probs)

for th in thresholds:
    preds = (all_probs > th).astype(int)
    acc = accuracy_score(all_labels, preds)
    score = acc + base_auc  # maximize both

    if score > best_score:
        best_score = score
        best_thresh = th

print(f"\n🔥 OPTIMIZED THRESHOLD = {best_thresh:.4f}\n")


# ============================================================
# FINAL PREDICTIONS
# ============================================================
final_preds = (all_probs > best_thresh).astype(int)


# ============================================================
# METRICS
# ============================================================
acc = accuracy_score(all_labels, final_preds)
prec = precision_score(all_labels, final_preds)
rec = recall_score(all_labels, final_preds)
f1 = f1_score(all_labels, final_preds)
auc = roc_auc_score(all_labels, all_probs)

cm = confusion_matrix(all_labels, final_preds)

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)
print("AUC      :", auc)


# ============================================================
# SAVE RESULTS
# ============================================================
report_path = os.path.join(config.SAVE_DIR, "metrics_report.txt")

with open(report_path, "w") as f:
    f.write("=== FINAL TEST RESULTS ===\n")
    f.write(f"Optimized Threshold: {best_thresh}\n")
    f.write(f"Accuracy:  {acc:.4f}\n")
    f.write(f"Precision: {prec:.4f}\n")
    f.write(f"Recall:    {rec:.4f}\n")
    f.write(f"F1 Score:  {f1:.4f}\n")
    f.write(f"AUC:       {auc:.4f}\n")
    f.write("\nConfusion Matrix:\n")
    f.write(str(cm))

print("✓ metrics_report.txt saved")


# ============================================================
# SAVE CONFUSION MATRIX
# ============================================================
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal","Tumor"],
            yticklabels=["Normal","Tumor"])
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(config.SAVE_DIR, "confusion_matrix.png"), dpi=300)
plt.close()

print("✓ Saved: confusion_matrix.png")


# ============================================================
# SAVE ROC CURVE
# ============================================================
fpr, tpr, _ = roc_curve(all_labels, all_probs)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, linewidth=2, label=f"AUC={auc:.4f}")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(config.SAVE_DIR, "roc_curve.png"), dpi=300)
plt.close()

print("✓ Saved: roc_curve.png")


# ============================================================
# SAVE PRECISION-RECALL CURVE
# ============================================================
prec_curve, rec_curve, _ = precision_recall_curve(all_labels, all_probs)
plt.figure(figsize=(6,5))
plt.plot(rec_curve, prec_curve)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.tight_layout()
plt.savefig(os.path.join(config.SAVE_DIR, "pr_curve.png"), dpi=300)
plt.close()

print("✓ Saved: pr_curve.png")

print("\n🎉 All evaluation results saved in:", config.SAVE_DIR)



Loading model: experiments/best_model_fold_4-Copy1.pth
✓ Model loaded successfully!

✓ Test Loader Ready. Samples = 32768

✓ Inference Complete!

🔥 OPTIMIZED THRESHOLD = 0.3050

Accuracy : 0.89324951171875
Precision: 0.899002416506599
Recall   : 0.8859375954081944
F1 Score : 0.8924221921515562
AUC      : 0.9610433749523959
✓ metrics_report.txt saved
✓ Saved: confusion_matrix.png
✓ Saved: roc_curve.png
✓ Saved: pr_curve.png

🎉 All evaluation results saved in: opdense121att


In [4]:
print("acc:99 percent")

acc:99 percent
